<a href="https://colab.research.google.com/github/huiningjiao02-ship-it/NeuroMM-2026-Track1-EEG-IED-Detection/blob/main/NeuroMM_2026_Track1_EEG_IED_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("RUNNING CELL 1: SETUP_ENV_FINAL")

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import sys
import subprocess
import shutil
import glob
import json
import pandas as pd
import numpy as np

REPO_DIR = "/content/NeuroMM-2026_Baseline"
WORK_DIR = "/content/neuromm_work"

os.makedirs(WORK_DIR, exist_ok=True)

print("REPO_DIR:", REPO_DIR)
print("WORK_DIR:", WORK_DIR)

if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/NeuroMM-Org/NeuroMM-2026_Baseline.git", REPO_DIR],
        check=True
    )
else:
    print("官方 baseline 已经存在。")

os.chdir(REPO_DIR)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "."],
    check=True
)

print("第 1 块完成：环境安装完成。")

RUNNING CELL 1: SETUP_ENV_FINAL
Mounted at /content/drive
REPO_DIR: /content/NeuroMM-2026_Baseline
WORK_DIR: /content/neuromm_work
第 1 块完成：环境安装完成。


In [2]:
print("RUNNING CELL 2: DATA_PREP_FINAL")

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import glob
import shutil
import tarfile
import zipfile
import json
import numpy as np
from collections import Counter

ROOT = "/content/drive/MyDrive"
REPO_DIR = "/content/NeuroMM-2026_Baseline"
WORK_DIR = "/content/neuromm_work"

# 清理旧的错误解压结果
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

LOCAL_TAR_DIR = f"{WORK_DIR}/local_tars"
TRAIN_DATA_DIR = f"{WORK_DIR}/train_data"
CANDIDATE_DATA_DIR = f"{WORK_DIR}/candidate_data"
CANDIDATE_WORK_DIR = f"{WORK_DIR}/candidate_track1"

os.makedirs(LOCAL_TAR_DIR, exist_ok=True)
os.makedirs(TRAIN_DATA_DIR, exist_ok=True)
os.makedirs(CANDIDATE_DATA_DIR, exist_ok=True)
os.makedirs(CANDIDATE_WORK_DIR, exist_ok=True)

def find_file(filename):
    matches = []
    for root, dirs, files in os.walk(ROOT):
        if filename in files:
            matches.append(os.path.join(root, filename))

    if len(matches) == 0:
        raise FileNotFoundError(f"没有找到文件：{filename}")

    matches = sorted(matches, key=lambda p: os.path.getsize(p), reverse=True)
    print(f"{filename} 找到：{matches[0]}")
    return matches[0]

train_csv = find_file("neuromm2026_train_val.csv")
candidate_ids = find_file("candidate_ids.txt")
train_eeg_tar_drive = find_file("train_eeg.tar")
candidate_eeg_tar_drive = find_file("candidate_eeg.tar")

def copy_to_local(src, dst):
    src_size = os.path.getsize(src)

    print("=" * 80)
    print("开始复制到 Colab 本地：", src)
    print("目标文件：", dst)
    print("文件大小：", f"{src_size / 1024 / 1024 / 1024:.2f} GB")

    with open(src, "rb") as fsrc, open(dst, "wb") as fdst:
        shutil.copyfileobj(fsrc, fdst, length=1024 * 1024 * 64)

    dst_size = os.path.getsize(dst)

    if dst_size != src_size:
        raise RuntimeError(
            f"复制不完整：源文件 {src_size} bytes，目标文件 {dst_size} bytes。请重新运行本 cell。"
        )

    print("复制完成：", dst)
    return dst

train_eeg_tar = copy_to_local(
    train_eeg_tar_drive,
    f"{LOCAL_TAR_DIR}/train_eeg.tar"
)

candidate_eeg_tar = copy_to_local(
    candidate_eeg_tar_drive,
    f"{LOCAL_TAR_DIR}/candidate_eeg.tar"
)

def list_npy(root_dir):
    return glob.glob(f"{root_dir}/**/*.npy", recursive=True)

def list_inner_archives(root_dir):
    archive_files = []
    patterns = [
        f"{root_dir}/**/*.tar",
        f"{root_dir}/**/*.tar.gz",
        f"{root_dir}/**/*.tgz",
        f"{root_dir}/**/*.zip",
    ]
    for pattern in patterns:
        archive_files.extend(glob.glob(pattern, recursive=True))
    return archive_files

def extract_archive(archive_path, out_dir):
    print("正在解压：", archive_path)

    if archive_path.endswith(".zip"):
        with zipfile.ZipFile(archive_path, "r") as z:
            z.extractall(out_dir)
    else:
        with tarfile.open(archive_path, "r:*") as tar:
            tar.extractall(out_dir)

def show_dir_tree(out_dir, max_level=5):
    print("当前解压目录结构：")
    for root, dirs, files in os.walk(out_dir):
        level = root.replace(out_dir, "").count(os.sep)
        if level <= max_level:
            print(root, "文件数：", len(files), "文件示例：", files[:10])

def recursive_extract(first_archive, out_dir, name, max_rounds=8):
    print("=" * 80)
    print("开始处理：", name)
    print("第一层压缩包：", first_archive)

    extract_archive(first_archive, out_dir)

    extracted_archives = set()

    for round_id in range(max_rounds):
        npy_files = list_npy(out_dir)
        print(f"第 {round_id + 1} 轮检查，找到 .npy 文件数量：", len(npy_files))

        if len(npy_files) > 0:
            print(name, "已经找到 EEG .npy 文件。")
            return npy_files

        inner_archives = list_inner_archives(out_dir)

        inner_archives = [
            p for p in inner_archives
            if p not in extracted_archives
        ]

        print("发现尚未解压的下一层压缩包数量：", len(inner_archives))

        if len(inner_archives) == 0:
            show_dir_tree(out_dir)
            raise RuntimeError(
                f"{name} 解压后没有找到 .npy，也没有找到下一层 tar/zip。"
            )

        for archive in inner_archives:
            extracted_archives.add(archive)
            extract_archive(archive, os.path.dirname(archive))

    raise RuntimeError(f"{name} 递归解压 {max_rounds} 轮后仍然没有找到 .npy。")

train_files = recursive_extract(
    train_eeg_tar,
    TRAIN_DATA_DIR,
    "训练集 + 验证集 EEG"
)

candidate_files = recursive_extract(
    candidate_eeg_tar,
    CANDIDATE_DATA_DIR,
    "candidate EEG"
)

def find_main_eeg_dir(files, name):
    parent_dirs = [os.path.dirname(p) for p in files]
    counter = Counter(parent_dirs)
    main_dir, count = counter.most_common(1)[0]
    print(f"{name} EEG 主目录：", main_dir)
    print(f"{name} EEG 文件数量：", count)
    return main_dir

train_eeg_dir = find_main_eeg_dir(train_files, "训练集")
candidate_eeg_dir = find_main_eeg_dir(candidate_files, "candidate")

x_train = np.load(train_files[0])
x_candidate = np.load(candidate_files[0])

print("训练 EEG 样本 shape：", x_train.shape)
print("candidate EEG 样本 shape：", x_candidate.shape)

dataset_root = f"{REPO_DIR}/neuromm26_datasets"
repo_annotations_dir = f"{dataset_root}/annotations"
repo_processed_dir = f"{dataset_root}/processed"
repo_features_dir = f"{repo_processed_dir}/features"
repo_eeg_dir = f"{repo_features_dir}/eeg"

os.makedirs(repo_annotations_dir, exist_ok=True)

shutil.copy(
    train_csv,
    f"{repo_annotations_dir}/neuromm2026_train_val.csv"
)

if os.path.islink(repo_processed_dir) or os.path.isfile(repo_processed_dir):
    os.remove(repo_processed_dir)
elif os.path.isdir(repo_processed_dir):
    shutil.rmtree(repo_processed_dir)

os.makedirs(repo_features_dir, exist_ok=True)

if os.path.exists(repo_eeg_dir):
    if os.path.islink(repo_eeg_dir) or os.path.isfile(repo_eeg_dir):
        os.remove(repo_eeg_dir)
    else:
        shutil.rmtree(repo_eeg_dir)

os.symlink(train_eeg_dir, repo_eeg_dir)

candidate_ids_dst = f"{CANDIDATE_WORK_DIR}/candidate_ids.txt"
candidate_eeg_dst = f"{CANDIDATE_WORK_DIR}/eeg"
candidate_processed_eeg_dst = f"{CANDIDATE_WORK_DIR}/processed/features/eeg"

if os.path.exists(candidate_ids_dst):
    os.remove(candidate_ids_dst)

shutil.copy(candidate_ids, candidate_ids_dst)

if os.path.islink(candidate_eeg_dst) or os.path.isfile(candidate_eeg_dst):
    os.remove(candidate_eeg_dst)
elif os.path.isdir(candidate_eeg_dst):
    shutil.rmtree(candidate_eeg_dst)

os.symlink(candidate_eeg_dir, candidate_eeg_dst)

os.makedirs(os.path.dirname(candidate_processed_eeg_dst), exist_ok=True)

if os.path.islink(candidate_processed_eeg_dst) or os.path.isfile(candidate_processed_eeg_dst):
    os.remove(candidate_processed_eeg_dst)
elif os.path.isdir(candidate_processed_eeg_dst):
    shutil.rmtree(candidate_processed_eeg_dst)

os.symlink(candidate_eeg_dir, candidate_processed_eeg_dst)

paths = {
    "repo_dir": REPO_DIR,
    "work_dir": WORK_DIR,
    "train_csv": train_csv,
    "candidate_ids": candidate_ids,
    "train_eeg_dir": train_eeg_dir,
    "candidate_eeg_dir": candidate_eeg_dir,
    "candidate_work_dir": CANDIDATE_WORK_DIR,
    "repo_eeg_dir": repo_eeg_dir,
}

with open(f"{WORK_DIR}/paths.json", "w", encoding="utf-8") as f:
    json.dump(paths, f, ensure_ascii=False, indent=2)

print("=" * 80)
print("baseline 训练 EEG 路径：", repo_eeg_dir)
print("candidate 提交 EEG 路径：", candidate_eeg_dst)
print("baseline 训练 EEG 文件数量：", len(glob.glob(f"{repo_eeg_dir}/*.npy")))
print("candidate 提交 EEG 文件数量：", len(glob.glob(f"{candidate_eeg_dst}/*.npy")))
print("第 2 块完成：数据解压和路径整理全部完成。")

RUNNING CELL 2: DATA_PREP_FINAL
Mounted at /content/drive
neuromm2026_train_val.csv 找到：/content/drive/MyDrive/NeuroMM2026/neuromm2026_train_val.csv
candidate_ids.txt 找到：/content/drive/MyDrive/NeuroMM2026/candidate_ids.txt
train_eeg.tar 找到：/content/drive/MyDrive/NeuroMM2026/train_eeg.tar
candidate_eeg.tar 找到：/content/drive/MyDrive/NeuroMM2026/candidate_eeg.tar
开始复制到 Colab 本地： /content/drive/MyDrive/NeuroMM2026/train_eeg.tar
目标文件： /content/neuromm_work/local_tars/train_eeg.tar
文件大小： 5.51 GB
复制完成： /content/neuromm_work/local_tars/train_eeg.tar
开始复制到 Colab 本地： /content/drive/MyDrive/NeuroMM2026/candidate_eeg.tar
目标文件： /content/neuromm_work/local_tars/candidate_eeg.tar
文件大小： 4.34 GB
复制完成： /content/neuromm_work/local_tars/candidate_eeg.tar
开始处理： 训练集 + 验证集 EEG
第一层压缩包： /content/neuromm_work/local_tars/train_eeg.tar
正在解压： /content/neuromm_work/local_tars/train_eeg.tar


/tmp/ipykernel_447/4244915830.py:105: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(out_dir)


第 1 轮检查，找到 .npy 文件数量： 25426
训练集 + 验证集 EEG 已经找到 EEG .npy 文件。
开始处理： candidate EEG
第一层压缩包： /content/neuromm_work/local_tars/candidate_eeg.tar
正在解压： /content/neuromm_work/local_tars/candidate_eeg.tar
第 1 轮检查，找到 .npy 文件数量： 20000
candidate EEG 已经找到 EEG .npy 文件。
训练集 EEG 主目录： /content/neuromm_work/train_data/processed/features/eeg
训练集 EEG 文件数量： 25426
candidate EEG 主目录： /content/neuromm_work/candidate_data/candidate/eeg
candidate EEG 文件数量： 20000
训练 EEG 样本 shape： (29, 2000)
candidate EEG 样本 shape： (29, 2000)
baseline 训练 EEG 路径： /content/NeuroMM-2026_Baseline/neuromm26_datasets/processed/features/eeg
candidate 提交 EEG 路径： /content/neuromm_work/candidate_track1/eeg
baseline 训练 EEG 文件数量： 25426
candidate 提交 EEG 文件数量： 20000
第 2 块完成：数据解压和路径整理全部完成。


In [3]:
print("RUNNING CELL 3: CHECK_GPU_AND_DATA_FINAL")

import os
import json
import glob
import numpy as np
import pandas as pd
import torch

WORK_DIR = "/content/neuromm_work"

with open(f"{WORK_DIR}/paths.json", "r", encoding="utf-8") as f:
    paths = json.load(f)

print("CUDA 是否可用：", torch.cuda.is_available())

if torch.cuda.is_available():
    print("当前 GPU：", torch.cuda.get_device_name(0))
else:
    print("当前没有 GPU。建议在 Runtime / 代码执行程序中切换到 GPU。")

train_csv = paths["train_csv"]
repo_eeg_dir = paths["repo_eeg_dir"]
candidate_work_dir = paths["candidate_work_dir"]

df = pd.read_csv(train_csv)

print("标签表形状：", df.shape)
print("标签表前 5 行：")
display(df.head())

print("split 分布：")
print(df["split"].value_counts())

print("label 分布：")
print(df["label"].value_counts())

eeg_files = glob.glob(f"{repo_eeg_dir}/*.npy")
print("训练 EEG 文件数量：", len(eeg_files))

if len(eeg_files) == 0:
    raise RuntimeError("训练 EEG 文件数量为 0。")

x = np.load(eeg_files[0])
print("示例训练 EEG shape：", x.shape)
print("示例训练 EEG dtype：", x.dtype)

candidate_ids_path = f"{candidate_work_dir}/candidate_ids.txt"
candidate_eeg_dir = f"{candidate_work_dir}/eeg"

with open(candidate_ids_path, "r") as f:
    candidate_id_lines = [line.strip() for line in f.readlines() if line.strip()]

candidate_eeg_files = glob.glob(f"{candidate_eeg_dir}/*.npy")

print("candidate id 数量：", len(candidate_id_lines))
print("candidate EEG 文件数量：", len(candidate_eeg_files))

if len(candidate_eeg_files) == 0:
    raise RuntimeError("candidate EEG 文件数量为 0。")

print("第 3 块完成：GPU 和数据检查完成。")

RUNNING CELL 3: CHECK_GPU_AND_DATA_FINAL
CUDA 是否可用： True
当前 GPU： NVIDIA L4
标签表形状： (25426, 5)
标签表前 5 行：


,sample_id,split,label,label_type,subject_id
0,DA001003_0_2000_500__0,train,0,0,DA001003
1,DA001003_100000_102000_500__0,train,0,0,DA001003
2,DA001003_10000_12000_500__0,train,0,0,DA001003
3,DA001003_102000_104000_500__0,train,0,0,DA001003
4,DA001003_104000_106000_500__0,train,0,0,DA001003


split 分布：
split
train    20298
val       5128
Name: count, dtype: int64
label 分布：
label
0    22912
1     2514
Name: count, dtype: int64
训练 EEG 文件数量： 25426
示例训练 EEG shape： (29, 2000)
示例训练 EEG dtype： float32
candidate id 数量： 20000
candidate EEG 文件数量： 20000
第 3 块完成：GPU 和数据检查完成。


In [4]:
print("RUNNING CELL 4: TRAIN_TRACK1_FINAL")

import os
import sys
import json
import subprocess
import pandas as pd

WORK_DIR = "/content/neuromm_work"

with open(f"{WORK_DIR}/paths.json", "r", encoding="utf-8") as f:
    paths = json.load(f)

REPO_DIR = paths["repo_dir"]

os.chdir(REPO_DIR)

MODEL_NAME = "legacy/tcnet_eeg"
SEED = 0
LR = "5e-4"
BATCH_SIZE = 64
EPOCHS = 30

cmd = [
    sys.executable,
    "-m",
    "neuromm26_baseline.tools.train",
    "--config",
    "configs/train_eeg_only.yaml",
    "--model-name",
    MODEL_NAME,
    "--seed",
    str(SEED),
    "--learning-rate",
    LR,
    "--batch-size",
    str(BATCH_SIZE),
    "--epochs",
    str(EPOCHS),
]

print("开始训练模型：", MODEL_NAME)
print("运行命令：", " ".join(cmd))

subprocess.run(cmd, cwd=REPO_DIR, check=True)

print("训练完成，开始汇总结果。")

subprocess.run(
    [sys.executable, "scripts/aggregate_results.py"],
    cwd=REPO_DIR,
    check=True
)

summary_csv = f"{REPO_DIR}/neuromm26_results/metrics/baseline_summary.csv"

if os.path.exists(summary_csv):
    df = pd.read_csv(summary_csv)
    display(df)
else:
    print("没有找到 baseline_summary.csv。")

print("第 4 块完成：模型训练完成。")

RUNNING CELL 4: TRAIN_TRACK1_FINAL
开始训练模型： legacy/tcnet_eeg
运行命令： /usr/bin/python3 -m neuromm26_baseline.tools.train --config configs/train_eeg_only.yaml --model-name legacy/tcnet_eeg --seed 0 --learning-rate 5e-4 --batch-size 64 --epochs 30
训练完成，开始汇总结果。


,model,lr,n_seeds,seeds,auprc_mean,auprc_std,binary_f1_mean,binary_f1_std,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std
0,legacy/tcnet_eeg,0.0005,1,[0],0.728792,0.0,0.661509,0.0,0.811778,0.0,0.933212,0.0,0.931747,0.0,0.825991,0.0


第 4 块完成：模型训练完成。


In [5]:
print("RUNNING CELL 5: MAKE_SUBMISSION_FINAL")

import os
import sys
import glob
import json
import zipfile
import shutil
import subprocess

WORK_DIR = "/content/neuromm_work"

with open(f"{WORK_DIR}/paths.json", "r", encoding="utf-8") as f:
    paths = json.load(f)

REPO_DIR = paths["repo_dir"]
CANDIDATE_WORK_DIR = paths["candidate_work_dir"]

os.chdir(REPO_DIR)

ckpt_list = glob.glob(
    f"{REPO_DIR}/neuromm26_results/checkpoints/**/best.pt",
    recursive=True
)

if len(ckpt_list) == 0:
    raise FileNotFoundError("没有找到 best.pt，请先完成第 4 块训练。")

ckpt_list = sorted(ckpt_list, key=os.path.getmtime)
best_ckpt = ckpt_list[-1]

exp_name = os.path.basename(os.path.dirname(best_ckpt))
config_json = f"{REPO_DIR}/neuromm26_results/metrics/{exp_name}_config.json"

if not os.path.exists(config_json):
    raise FileNotFoundError(f"没有找到 config 文件：{config_json}")

submission_csv = f"{REPO_DIR}/submission.csv"

cmd = [
    sys.executable,
    "-m",
    "neuromm26_baseline.tools.predict_candidate_test1",
    "--checkpoint",
    best_ckpt,
    "--config",
    config_json,
    "--candidate-dir",
    CANDIDATE_WORK_DIR,
    "--out",
    submission_csv,
]

print("使用 checkpoint：", best_ckpt)
print("使用 config：", config_json)
print("candidate 目录：", CANDIDATE_WORK_DIR)
print("开始生成 submission.csv。")
print("运行命令：", " ".join(cmd))

subprocess.run(cmd, cwd=REPO_DIR, check=True)

zip_path = f"{REPO_DIR}/sample_submission.zip"

if os.path.exists(zip_path):
    os.remove(zip_path)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(submission_csv, arcname="submission.csv")

drive_output_dir = os.path.dirname(paths["train_csv"])
drive_zip_path = f"{drive_output_dir}/sample_submission.zip"

shutil.copy(zip_path, drive_zip_path)

print("第 5 块完成：提交文件生成完成。")
print("你要上传到 CodaBench 的文件是：", drive_zip_path)

RUNNING CELL 5: MAKE_SUBMISSION_FINAL
使用 checkpoint： /content/NeuroMM-2026_Baseline/neuromm26_results/checkpoints/neuromm26_train_eeg_only__legacy-tcnet_eeg__seed0__lr0.0005/best.pt
使用 config： /content/NeuroMM-2026_Baseline/neuromm26_results/metrics/neuromm26_train_eeg_only__legacy-tcnet_eeg__seed0__lr0.0005_config.json
candidate 目录： /content/neuromm_work/candidate_track1
开始生成 submission.csv。
运行命令： /usr/bin/python3 -m neuromm26_baseline.tools.predict_candidate_test1 --checkpoint /content/NeuroMM-2026_Baseline/neuromm26_results/checkpoints/neuromm26_train_eeg_only__legacy-tcnet_eeg__seed0__lr0.0005/best.pt --config /content/NeuroMM-2026_Baseline/neuromm26_results/metrics/neuromm26_train_eeg_only__legacy-tcnet_eeg__seed0__lr0.0005_config.json --candidate-dir /content/neuromm_work/candidate_track1 --out /content/NeuroMM-2026_Baseline/submission.csv
第 5 块完成：提交文件生成完成。
你要上传到 CodaBench 的文件是： /content/drive/MyDrive/NeuroMM2026/sample_submission.zip
